<a href="https://colab.research.google.com/github/sunyuhongwr/-/blob/main/deep_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第四讲：用深度学习给睡眠分期

**目标：** 亲手搭建并训练几种最常用的神经网络模块，看看它们各自擅长什么。

| 模块 | 一句话理解 | 今天用它做什么 |
|---|---|---|
| 全连接层 `nn.Linear` | 把所有输入加权求和 | 最简单的基线 |
| 卷积层 `nn.Conv1d`（CNN） | 一个小模板沿着信号滑动，寻找特定的波形 | 从 30 秒原始脑电里自动提取特征 |
| 循环层 `nn.RNN`、`nn.LSTM`、`nn.GRU` | 按时间顺序读入，并记住之前看到的内容 | 利用前后 epoch 的信息 |
| `nn.Transformer` | 让序列里每个位置直接"看到"其他所有位置 | 同上，另一种做法 |

**和第二讲机器学习的区别：** 第二讲我们先手工算好特征，再交给模型。现在直接把**原始脑电**（C4-M1）交给神经网络，让它自己学特征。

**怎么用：** 点一下代码格，按 `Shift + Enter` 运行，从上到下按顺序来。标有 ✏️ 的地方可以动手改。

**关于 GPU：** 本讲的模型很小，不开 GPU 也能在一两分钟内跑完全部内容。想体验 GPU 的话，可以在菜单栏"runtime" → "change runtime type"里把硬件加速器选为 **T4 GPU**（切换后需要从头重新运行）。

## 第 0 步：准备工作

运行下面这格，它会载入工具并下载数据（约 93 MB，需要十几秒）。看到"准备完成"就可以继续。

In [ ]:
# 导入依赖
import os, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, recall_score, ConfusionMatrixDisplay

# 数据下载
DRIVE_IDS = {
    "sleep_demo_raw.npz": "1Q9RLyw1ibUfToWGITAUsvyB3WM175nSh",
    "cnn_pretrained.pt": "1xrW4zufEu-WJ8XJcz_6J-qpw8fijrOG2",  # CNN 预训练权重
}
for name, file_id in DRIVE_IDS.items():
    if os.path.exists(name) or (name.endswith(".pt") and not file_id):
        continue
    if file_id:
        import gdown
        gdown.download(id=file_id, output=name, quiet=False)
    else:
        from google.colab import files
        print(f"请上传文件 {name}")
        files.upload()

device = "cuda" if torch.cuda.is_available() else "cpu"
STAGES = ["Wake", "N1", "N2", "N3", "REM"]
COLORS = {"Wake": "#D0922A", "N1": "#3DB09A", "N2": "#3D7EC9", "N3": "#63408F", "REM": "#BE4F3C"}
HYPNO_Y = {0: 4, 4: 3, 1: 2, 2: 1, 3: 0}
val_scores = {}  # 验证集准确率
predictions = {}  # 测试集预测


def loss_and_accuracy(model, inputs, targets, batch_size=512):
    """计算平均损失和准确率。"""
    model.eval()
    loss_fn = nn.CrossEntropyLoss(reduction="sum")
    total, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for i in range(0, len(inputs), batch_size):
            out = model(inputs[i:i + batch_size].to(device)).reshape(-1, 5)
            yb = targets[i:i + batch_size].to(device).reshape(-1)
            total += loss_fn(out, yb).item()
            correct += (out.argmax(dim=1) == yb).sum().item()
            count += len(yb)
    return total / count, correct / count


def train_model(model, inputs, targets, epochs, lr=1e-3, batch_size=128, val_inputs=None, val_targets=None):
    """逐轮训练并记录指标。"""
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = {"loss": [], "val_loss": [], "val_acc": []}
    for ep in range(epochs):
        t0 = time.time()
        model.train()
        order = torch.randperm(len(inputs))
        total = 0.0
        for i in range(0, len(order), batch_size):
            idx = order[i:i + batch_size]
            xb, yb = inputs[idx].to(device), targets[idx].to(device)
            out = model(xb)  # 前向传播
            loss = loss_fn(out.reshape(-1, 5), yb.reshape(-1))  # 计算损失
            optimizer.zero_grad()
            loss.backward()  # 反向传播
            optimizer.step()  # 更新参数
            total += loss.item() * len(idx)
        history["loss"].append(total / len(order))
        msg = f"第 {ep + 1:2d} 轮  训练 loss {history['loss'][-1]:.3f}"
        if val_inputs is not None:
            v_loss, v_acc = loss_and_accuracy(model, val_inputs, val_targets)
            history["val_loss"].append(v_loss)
            history["val_acc"].append(v_acc)
            msg += f"  验证 loss {v_loss:.3f}  验证准确率 {v_acc:.1%}"
        print(msg + f"  （{time.time() - t0:.0f} 秒）")
    return history


def predict(model, inputs, batch_size=512):
    """预测各 epoch 的分期。"""
    model.to(device).eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(inputs), batch_size):
            out.append(model(inputs[i:i + batch_size].to(device)).argmax(dim=-1).cpu())
    return torch.cat(out).numpy()


def record(name, val_pred, test_pred):
    """记录验证指标和测试集预测。"""
    acc = accuracy_score(y_val, val_pred)
    recall = recall_score(y_val, val_pred, average=None, labels=range(5), zero_division=0)
    val_scores[name] = acc
    predictions[name] = np.asarray(test_pred)
    print(f"{name}（验证集）：准确率 {acc:.1%}，宏 F1 {f1_score(y_val, val_pred, average='macro'):.2f}")
    print("各分期判对的比例：" + "，".join(f"{s} {r:.0%}" for s, r in zip(STAGES, recall)))


def plot_hypnogram(ax, epoch, labels, **style):
    hours = np.asarray(epoch) * 30 / 3600
    ax.step(hours, [HYPNO_Y[int(v)] for v in labels], where="post", **style)
    ax.set_yticks(range(5))
    ax.set_yticklabels(["N3", "N2", "N1", "REM", "Wake"])


class PositionalEncoding(nn.Module):
    """编码 epoch 在序列中的位置。"""
    def __init__(self, size, max_len=500):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, size, 2) * (-math.log(10000.0) / size))
        pe = torch.zeros(max_len, size)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[: x.shape[1]]


torch.manual_seed(0)
np.random.seed(0)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("准备完成。当前使用的设备：", "GPU" if device == "cuda" else "CPU（今天的模型很小，用 CPU 就够了）")

## 第 1 步：准备数据

和第二讲一样的 20 晚、一样的划分。不同的是，这次交给模型的不是特征表，而是**原始脑电**：每个 epoch 是 3000 个数字（30 秒 × 每秒 100 个点）。

- **训练集（13 人）**：模型从这里学习。
- **验证集（3 人）**：训练过程中用它观察模型学得怎么样，并用来比较不同的设置。
- **测试集（4 人）**：全程不参与任何选择，最后一步才用。

In [ ]:
data = np.load("sleep_demo_raw.npz")
X = data["X"].astype(np.float32)  # EEG 形状：(epoch 数, 3000)
y = data["y"].astype(np.int64)  # 专家分期标签
subject = data["subject"]  # 被试编号
epoch_idx = data["epoch"]  # 整晚 epoch 索引

X = np.clip(X / 30.0, -10, 10)  # 缩放并截断振幅

is_train, is_val, is_test = (data["split"] == s for s in ("train", "val", "test"))
X_train, y_train = torch.tensor(X[is_train]).unsqueeze(1), torch.tensor(y[is_train])  # EEG 张量：(epoch 数, 1, 3000)
X_val, y_val_t = torch.tensor(X[is_val]).unsqueeze(1), torch.tensor(y[is_val])
X_test = torch.tensor(X[is_test]).unsqueeze(1)
y_val, y_test = y[is_val], y[is_test]

for name, mask in (("训练集", is_train), ("验证集", is_val), ("测试集", is_test)):
    print(f"{name}：{len(np.unique(subject[mask]))} 人，{mask.sum()} 个 epoch")

## 第 2 步：模块一，全连接网络

最基础的神经网络模块是**全连接层** `nn.Linear`：把输入的每个数字乘上一个权重再加起来。我们把 3000 个点直接接到全连接层上，看看效果。

In [ ]:
mlp = nn.Sequential(
    nn.Flatten(),  # 展平输入
    nn.Linear(3000, 128),  # 全连接：3000 → 128
    nn.ReLU(),  # ReLU 激活
    nn.Linear(128, 5),  # 输出：5 类分期得分
)

history = train_model(mlp, X_train, y_train, epochs=5, val_inputs=X_val, val_targets=y_val_t)
record("MLP (raw EEG)", predict(mlp, X_val), predict(mlp, X_test))

**观察：** 训练 loss 一直在下降，说明网络越来越会"背"训练数据；但验证 loss 没有跟着降，验证集准确率也很低。在训练数据上进步，不等于在新的人身上也进步。

**为什么全连接层不行？** 它把"第 517 个点"和"第 518 个点"当成毫不相干的两个输入。可是一个慢波出现在第 3 秒还是第 20 秒，对判断分期来说是一回事。我们需要一种**不在乎波形出现在哪里**的模块。

## 第 3 步：模块二，卷积神经网络（CNN）

**卷积层** `nn.Conv1d` 的想法：准备一批很短的小模板（称为卷积核或 filter），让每个模板沿着信号从头滑到尾，哪里的波形和模板相似，哪里的输出就大。

- 模板的形状不是人设计的，而是训练中**学出来**的。
- 同一个模板在整段信号上重复使用，所以波形出现在哪里都能被找到。

下面的网络有三层卷积。第一层用 32 个长度为 0.5 秒的模板。

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # 32 个卷积核，核宽 50，步长 6
            nn.Conv1d(1, 32, kernel_size=50, stride=6), nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(8),  # 最大池化，窗口 8
            nn.Dropout(0.3),
            # 第 2、3 层卷积
            nn.Conv1d(32, 64, kernel_size=8, padding=4), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=8, padding=4), nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(4),
            nn.AdaptiveAvgPool1d(1),  # 时间维平均池化
            nn.Flatten(),
        )
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(64, 5))  # 分类头：64 → 5

    def forward(self, x):
        return self.classifier(self.features(x))

cnn = CNN()
print(f"MLP 的参数个数：{sum(p.numel() for p in mlp.parameters()):,}")
print(f"CNN 的参数个数：{sum(p.numel() for p in cnn.parameters()):,}")

注意 CNN 的参数比全连接网络**少得多**。

✏️ **选做：** 训练完成后，可以回来修改 `EPOCHS`（训练几轮）或 `LEARNING_RATE`（学习率，每次调整参数的步子有多大），重新运行这一格。比如试试 `LEARNING_RATE = 0.1` 或 `EPOCHS = 2`。

比较不同设置时请看**验证集**的曲线和成绩，不要用测试集来挑。

In [ ]:
EPOCHS = 6  # ✏️ 训练轮数
LEARNING_RATE = 0.001  # ✏️ 学习率
USE_PRETRAINED = False  # 加载预训练权重

torch.manual_seed(0)
cnn = CNN()
if USE_PRETRAINED and os.path.exists("cnn_pretrained.pt"):
    cnn.load_state_dict(torch.load("cnn_pretrained.pt", map_location="cpu"))
    print("已载入提前训练好的 CNN")
else:
    history = train_model(cnn, X_train, y_train, epochs=EPOCHS, lr=LEARNING_RATE, val_inputs=X_val, val_targets=y_val_t)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    rounds = range(1, len(history["loss"]) + 1)
    axes[0].plot(rounds, history["loss"], marker="o", color="#2376CF", label="training")
    axes[0].plot(rounds, history["val_loss"], marker="o", color="#BE4F3C", label="validation")
    axes[0].set_xlabel("Training round (epoch)"); axes[0].set_ylabel("Loss"); axes[0].set_title("Loss"); axes[0].legend()
    axes[1].plot(rounds, history["val_acc"], marker="o", color="#BE4F3C")
    axes[1].set_xlabel("Training round (epoch)"); axes[1].set_ylabel("Accuracy"); axes[1].set_title("Validation accuracy")
    axes[1].set_ylim(0, 1)
    plt.tight_layout(); plt.show()

record("CNN", predict(cnn, X_val), predict(cnn, X_test))

**观察：** 和全连接网络相比，CNN 在验证集上好了多少？没有人告诉它什么是 delta、什么是纺锤波。它和第二讲的随机森林相比如何，到最后一步在测试集上一起比较。

那它到底学到了什么？

## 第 4 步：打开模型看一看

### 4.1 第一层学到的模板

下图上排是第一层卷积学到的 8 个模板（每个长 0.5 秒），下排是每个模板对哪些频率最敏感。

In [ ]:
filters = cnn.features[0].weight.detach().cpu().numpy()[:, 0, :]  # 卷积核形状：(32, 50)
strongest = np.argsort(-np.abs(filters).sum(axis=1))[:8]  # 按绝对值和取前 8 个
freqs = np.fft.rfftfreq(400, d=1 / 100)

fig, axes = plt.subplots(2, 8, figsize=(14, 4), sharey="row")
for col, k in enumerate(strongest):
    axes[0, col].plot(np.arange(50) / 100, filters[k], color="#2376CF")
    axes[0, col].set_title(f"filter {k}", fontsize=9)
    axes[1, col].plot(freqs, np.abs(np.fft.rfft(filters[k], n=400)), color="#BE4F3C")
    axes[1, col].set_xlim(0, 40)
    axes[1, col].set_xlabel("Hz")
axes[0, 0].set_ylabel("Filter shape\n(0.5 s)")
axes[1, 0].set_ylabel("Frequency\nresponse")
plt.tight_layout(); plt.show()

**观察：** 只训练了几轮，模板的形状看起来还比较杂乱，但从下排的频率响应能看出一些分工：有的模板主要对几 Hz 以下的低频有反应，有的在 10 到 15 Hz 附近有一个峰。这让人联想到第二讲手工设计的"频段功率"特征。


### 4.2 一张"睡眠地图"

CNN 把每个 30 秒的 epoch 概括成 64 个数（称为 embedding）。把这 64 个数压缩到二维画出来，每个点是测试集里的一个 epoch，颜色是专家标注的分期。

In [ ]:
cnn.to(device).eval()
with torch.no_grad():
    emb_test = torch.cat([cnn.features(X_test[i:i + 512].to(device)).cpu() for i in range(0, len(X_test), 512)]).numpy()

xy = PCA(n_components=2).fit_transform(emb_test)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for k, s in enumerate(STAGES):
    ax.scatter(xy[y_test == k, 0], xy[y_test == k, 1], s=6, alpha=0.5, color=COLORS[s], label=s)
ax.set_xlabel("Component 1"); ax.set_ylabel("Component 2")
ax.set_title("Each dot is one 30-s epoch, as the CNN sees it")
ax.legend(markerscale=3)
plt.show()

**观察：** 离得近的点，表示 CNN 认为它们相似。相同分期的 epoch 大致聚在一起，N1 和 Wake、REM 混在一块，所以 N1 仍然最难分。

怎么办？想一想专家是怎么做的：遇到拿不准的 epoch，会**看看前后几分钟**是什么状态。而 CNN 每次只看一个孤立的 epoch。

## 第 5 步（进阶）：模块三，循环神经网络（RNN、LSTM、GRU）

**循环层**按时间顺序一个一个地读入序列，并且带着一份"记忆"往下传：读到第 10 个时，它还记得前 9 个的内容。

- `nn.RNN`：最简单的循环层，记忆容易随时间淡化。
- `nn.LSTM`：加了几道"门"来决定记住什么、忘掉什么，能记得更久。
- `nn.GRU`：LSTM 的简化版，参数更少。

**我们的做法（两步走）：**

1. 用刚才训练好的 CNN，把每个 epoch 变成 64 个数。
2. 把一晚上连续的 epoch 排成序列，交给循环层，让它结合前后文重新判断每个 epoch 的分期。

先做第 1 步，并把每位被试的数据切成长度为 40 个 epoch（20 分钟）的片段。

In [ ]:
SEQ_LEN = 40  # 序列长度：40 个 epoch

# 提取 64 维 CNN 特征
X_all = torch.tensor(X).unsqueeze(1)
cnn.to(device).eval()
with torch.no_grad():
    emb = torch.cat([cnn.features(X_all[i:i + 512].to(device)).cpu() for i in range(0, len(X_all), 512)])


def make_sequences(mask, step):
    """按被试切分 epoch 序列。"""
    seq_x, seq_y = [], []
    for s in np.unique(subject[mask]):
        idx = np.flatnonzero(subject == s)
        for start in range(0, len(idx) - SEQ_LEN + 1, step):
            seq_x.append(emb[idx[start:start + SEQ_LEN]])
            seq_y.append(torch.tensor(y[idx[start:start + SEQ_LEN]]))
    return torch.stack(seq_x), torch.stack(seq_y)


def predict_sequences(model, mask):
    """按被试分段预测 epoch 分期。"""
    model.to(device).eval()
    pred = np.zeros(len(y), dtype=int)
    with torch.no_grad():
        for s in np.unique(subject[mask]):
            idx = np.flatnonzero(subject == s)
            for start in range(0, len(idx), SEQ_LEN):
                part = idx[start:start + SEQ_LEN] if start + SEQ_LEN <= len(idx) else idx[-SEQ_LEN:]
                pred[part] = model(emb[part].unsqueeze(0).to(device)).argmax(dim=-1)[0].cpu().numpy()
    return pred[mask]


seq_x, seq_y = make_sequences(is_train, step=5)  # 训练窗口重叠
val_seq_x, val_seq_y = make_sequences(is_val, step=SEQ_LEN)  # 验证窗口不重叠
print("训练序列：", tuple(seq_x.shape), "即（序列个数, 每个序列 40 个 epoch, 每个 epoch 64 个数）")

下面定义序列模型。四种模块的用法几乎一样，只是换一个名字。

In [ ]:
class SequenceModel(nn.Module):
    def __init__(self, kind="lstm", hidden=64, bidirectional=False):
        super().__init__()
        self.kind = kind
        self.dropout = nn.Dropout(0.5)
        if kind == "rnn":
            self.seq = nn.RNN(64, hidden, batch_first=True, bidirectional=bidirectional)
        elif kind == "lstm":
            self.seq = nn.LSTM(64, hidden, batch_first=True, bidirectional=bidirectional)
        elif kind == "gru":
            self.seq = nn.GRU(64, hidden, batch_first=True, bidirectional=bidirectional)
        elif kind == "transformer":
            layer = nn.TransformerEncoderLayer(d_model=64, nhead=4, dim_feedforward=128, dropout=0.1, batch_first=True)
            self.seq = nn.Sequential(PositionalEncoding(64), nn.TransformerEncoder(layer, num_layers=2))
        out_size = 64 if kind == "transformer" else hidden * (2 if bidirectional else 1)
        self.classifier = nn.Linear(out_size, 5)

    def forward(self, x):  # 输入：(序列数, 40, 64)
        x = self.dropout(x)
        h = self.seq(x) if self.kind == "transformer" else self.seq(x)[0]
        return self.classifier(h)  # 逐 epoch 的 5 类得分

训练一个 LSTM。因为输入已经被 CNN 压缩成很短的序列，训练只需要几秒钟。

In [ ]:
torch.manual_seed(0)
lstm = SequenceModel(kind="lstm")
_ = train_model(lstm, seq_x, seq_y, epochs=10, batch_size=64, val_inputs=val_seq_x, val_targets=val_seq_y)
record("CNN + LSTM", predict_sequences(lstm, is_val), predict_sequences(lstm, is_test))

**观察：** 和单独的 CNN 相比，验证集上哪些分期变化最明显？

### ✏️ 练习（选做）：换一种模块

修改下面的 `KIND` 和 `BIDIRECTIONAL`，每改一次运行一次，验证集成绩会自动记录下来。

- `KIND` 可以是 `"rnn"`、`"lstm"`、`"gru"` 或 `"transformer"`。
- `BIDIRECTIONAL = True` 表示**双向**：除了从前往后读，再从后往前读一遍，这样每个 epoch 既能参考过去也能参考未来。（对 transformer 无效，它本来就能看到整个序列。）

In [ ]:
KIND = "gru"  # ✏️ 模型：rnn / lstm / gru / transformer
BIDIRECTIONAL = False  # ✏️ 双向序列模型

torch.manual_seed(0)
model = SequenceModel(kind=KIND, bidirectional=BIDIRECTIONAL)
_ = train_model(model, seq_x, seq_y, epochs=10, batch_size=64, val_inputs=val_seq_x, val_targets=val_seq_y)
name = "CNN + " + ("Bi-" if BIDIRECTIONAL and KIND != "transformer" else "") + KIND.upper()
record(name, predict_sequences(model, is_val), predict_sequences(model, is_test))

双向模型能同时参考过去和未来。但如果要在睡眠过程中**实时**判断分期（比如用来控制声音刺激），还能用双向模型吗？

## 第 6 步：打开测试集，把所有模型放在一起比较

到这里为止，所有的比较和选择都只用了验证集。现在才第一次看测试集的 4 个人。

表中最后一行是第二讲的随机森林（8 个 EEG 特征），它和今天的模型用的是同样的 13 人训练、同样的 4 人测试、同样的 C4-M1 导联和五个分期。

In [ ]:
rows = {}
for name, p in predictions.items():
    rows[name] = [val_scores[name], accuracy_score(y_test, p), f1_score(y_test, p, average="macro")] + \
                 list(recall_score(y_test, p, average=None, labels=range(5), zero_division=0))
rows["Random forest, EEG features (Session 2)"] = [0.778, 0.772, 0.712, 0.84, 0.30, 0.92, 0.81, 0.69]  # 第二讲随机森林指标
table = pd.DataFrame(rows, index=["val accuracy", "test accuracy", "test macro-F1"] + [f"test {s}" for s in STAGES]).T
table = table.sort_values("test accuracy")

fig, ax = plt.subplots(figsize=(8, 0.5 * len(table) + 1.2))
bars = ax.barh(table.index, table["test accuracy"], color=["#9A9A9A" if "Session 2" in n else "#2376CF" for n in table.index])
ax.bar_label(bars, fmt="{:.1%}", padding=3)
ax.set_xlim(0, 1)
ax.set_xlabel("Test accuracy (4 people the models have never seen)")
plt.show()
table.round(2)

**观察：** CNN 和随机森林相比如何？加上序列模型之后呢？

最后看一位测试被试整晚的结果：专家、CNN，以及验证集上表现最好的"CNN + 序列模型"。

In [ ]:
who = "S17"  # ✏️ 绘图被试
seq_names = [n for n in predictions if n.startswith("CNN + ")]
best = max(seq_names, key=lambda n: val_scores[n])  # 验证准确率最高的序列模型
rows = subject[is_test] == who
order = np.argsort(epoch_idx[is_test][rows])
t = epoch_idx[is_test][rows][order]

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
plot_hypnogram(axes[0], t, y_test[rows][order], color="black", linewidth=1)
axes[0].set_title(f"{who}: human expert")
for ax, name, color in ((axes[1], "CNN", "#BE4F3C"), (axes[2], best, "#2376CF")):
    p = predictions[name][rows][order]
    plot_hypnogram(ax, t, p, color=color, linewidth=1)
    ax.set_title(f"{who}: {name} (agreement {accuracy_score(y_test[rows][order], p):.0%})")
axes[2].set_xlabel("Time (hours)")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay.from_predictions(y_test, predictions[best], display_labels=STAGES, normalize="true",
                                        values_format=".2f", cmap="Blues", colorbar=False, ax=ax)
ax.set_xlabel(best); ax.set_ylabel("Human expert")
plt.show()

**观察：** 单独的 CNN 每次只看一个 epoch，结果比较"毛躁"；加上序列模型后，hypnogram 更连贯。

## 小结

| 模块 | 擅长什么 | 今天的角色 |
|---|---|---|
| 全连接 `nn.Linear` | 输入是一组互不相关的数字（比如第二讲的特征表） | 直接处理原始信号效果很差 |
| 卷积 `nn.Conv1d` | 在信号或图像里找**局部的形状**，不在乎出现在哪里 | 直接从原始脑电学习特征 |
| 循环 `nn.RNN` / `nn.LSTM` / `nn.GRU` | 有**先后顺序**的数据，需要记住上下文 | 利用前后 epoch 的信息 |
| `nn.Transformer` | 同样处理序列，数据量大时优势更明显 | 序列模块的另一种选择 |

**三点总结：**

1. **模块要和数据的结构匹配。** 局部波形用卷积，时间上的前后关系用循环层或 Transformer。实际的模型往往是把它们**组合**起来。
2. **深度学习省掉了手工设计特征，但需要更多数据和计算。** 今天只用了 13 个人的数据来训练。实际工作中一般不需要自己从头训练，可以使用或微调别人在大规模数据上**预训练**好的模型，前提是它的导联、预处理和人群与你的数据相符。
3. **数据少、需要解释清楚模型依据什么做判断时，第二讲的传统方法仍然是很好的选择。**